# Report Generation — Stage A (generate)

Worked example: run a vision-language model (VLM) over a RadHarmony
dataset to **generate** chest-X-ray reports, then score them against the
reference reports with [RadEval](https://github.com/jbdel/RadEval).

Report-gen scoring runs as a **two-stage, two-environment** workflow because
the generative backbones and RadEval pin mutually-incompatible `transformers`
versions:

- **Stage A (this notebook)** — generate reports and dump
  `(sample_id, reference, hypothesis)` triples to a parquet. Runs in the
  backbone's venv (e.g. `chexagent_gen`, `transformers==4.40.0`). **No RadEval
  import.**
- **Stage B** — score the parquet with RadEval in its own venv. See
  [`score_reports_radeval.ipynb`](score_reports_radeval.ipynb).

The example is **dataset-agnostic**: any RadHarmony dataset built with
`output_report=True` works. The only model-specific piece is the generator
factory — swap `make_chexagent_generator` for your own
`(transform, report_generator)` to evaluate a different VLM.

**Install (in the backbone's own venv):**
```bash
uv pip install -e ".[chexagent_gen]"   # or .[maira2_gen]
```

## Configuration

Edit these to point at your data and pick a backbone/dataset. Paths are
placeholders — replace with your local dataset locations.

In [ ]:
# --- data ---
DATASET        = "openi"          # "openi" | "mimic" | "emory"
BASE_IMAGE_DIR = "/path/to/openi/"  # image root for the chosen dataset
REPORT_CSV     = None             # mimic: cxr-study-list.csv.gz; emory: reports.csv
METADATA_CSV   = None             # emory only: metadata.csv (image mapping)
OUTPUT_DIR     = "outputs/reportgen"

# --- model ---
BACKBONE       = "chexagent"      # "chexagent" | "maira2"
DEVICE         = "cuda:0"

# --- scoring options ---
REF_SECTION    = None             # None -> 'full' for openi, 'findings' otherwise
BATCH_SIZE     = 8
MAX_SAMPLES    = 50               # smoke test; set None for a full run
N_BOOTSTRAP    = 0
LIGHT          = False            # True -> no-LLM/no-API LIGHT_METRICS subset

# --- workflow ---
MODE           = "generate"       # "generate" (Stage A only) | "both" (one-env)
PAIRS_PARQUET  = None             # default: <OUTPUT_DIR>/pairs.parquet

# --- optional: restrict to a fixed cohort (apples-to-apples with a probe) ---
SUBSET_PARQUET = None             # parquet of IDs to keep; None disables
SUBSET_COL     = "study_id"       # column joining SUBSET_PARQUET to the harmonized df

## Load the generative backbone

Each backbone's extra pins an incompatible `transformers`, so we import only
the one we will actually call.

In [ ]:
if BACKBONE == "chexagent":
    from radharmony.evaluator.backbones import make_chexagent_generator
    transform, report_generator = make_chexagent_generator(device=DEVICE)
elif BACKBONE == "maira2":
    from radharmony.evaluator.backbones import make_maira2_generator
    transform, report_generator = make_maira2_generator(device=DEVICE)
else:
    raise ValueError(f"unknown backbone: {BACKBONE}")

## Build the dataset

Any RadHarmony dataset with `output_report=True` works. Each branch records
`(class, kwargs)` so an optional subset filter can rebuild it via
`harmonized_df=` (the dataset skips harmonization when that is passed).

In [ ]:
if DATASET == "openi":
    from radharmony.dataset import OpenICXRDataset as DSCls
    ds_kwargs = dict(
        base_image_dir=BASE_IMAGE_DIR,   # root w/ ecgen-radiology/ + images/
        transform=transform,
        output_report=True,
    )
    ref_section = REF_SECTION or "full"
elif DATASET == "emory":
    from radharmony.dataset import EmoryCXRDataset as DSCls
    ds_kwargs = dict(
        base_image_dir=BASE_IMAGE_DIR,   # EmoryCXRv2 DEID_PNG root
        csv_path=METADATA_CSV,           # metadata.csv
        report_csv_path=REPORT_CSV,      # reports.csv (de-id report text)
        transform=transform,
        output_report=True,
    )
    ref_section = REF_SECTION or "findings"
else:  # mimic — DICOM variant only; MIMIC-CXR-JPG has no reports
    from radharmony.dataset import MIMICCXRDataset as DSCls
    ds_kwargs = dict(
        base_image_dir=BASE_IMAGE_DIR,
        report_csv_path=REPORT_CSV,      # cxr-study-list.csv.gz
        transform=transform,
        output_report=True,
    )
    ref_section = REF_SECTION or "findings"

ds = DSCls(**ds_kwargs)

### (Optional) Restrict to a fixed cohort

To compare a generator against another model on **exactly the same studies**,
filter the harmonized DataFrame to a set of IDs from a parquet, then rebuild
the dataset from the filtered frame. Skip this cell if `SUBSET_PARQUET is None`.

In [ ]:
if SUBSET_PARQUET:
    import pandas as pd
    keep = set(pd.read_parquet(SUBSET_PARQUET)[SUBSET_COL].astype(str))
    hdf = ds._get_harmonized_df()
    if SUBSET_COL not in hdf.columns:
        raise KeyError(
            f"{SUBSET_COL!r} not in harmonized df columns "
            f"{list(hdf.columns)} — set a valid SUBSET_COL."
        )
    before = len(hdf)
    hdf = hdf[hdf[SUBSET_COL].astype(str).isin(keep)].copy()
    print(f"subset: {before} -> {len(hdf)} rows ({len(keep)} ids)")
    ds = DSCls(**ds_kwargs, harmonized_df=hdf)

### (Optional) Smoke-test prefilter (MIMIC only)

For a quick smoke run against a **partial** MIMIC-CXR mirror, prefilter the
harmonized frame to rows whose image file exists on disk and truncate to
`MAX_SAMPLES`. This makes the dataloader iterate exactly `MAX_SAMPLES` items
and tolerates a sparse local mirror. Skip unless `DATASET == "mimic"` with a
small `MAX_SAMPLES`.

In [ ]:
import os

if DATASET == "mimic" and MAX_SAMPLES:
    hdf = ds._get_harmonized_df()
    # MIMIC patient dirs are named p<subject_id> (e.g. p10000032) but the
    # harmonizer stores subject_id verbatim (10000032). Strip the leading
    # 'p' when building the mirrored set. Listing pXX dirs is far cheaper
    # than tens of thousands of exists() calls over NFS.
    mirrored = set()
    for pxx in os.listdir(BASE_IMAGE_DIR):
        pxx_dir = os.path.join(BASE_IMAGE_DIR, pxx)
        if os.path.isdir(pxx_dir):
            try:
                for name in os.listdir(pxx_dir):
                    mirrored.add(name[1:] if name.startswith("p") else name)
            except OSError:
                pass
    before = len(hdf)
    if "patient_id" in hdf.columns and mirrored:
        hdf = hdf[hdf["patient_id"].astype(str).isin(mirrored)].copy()
    print(f"smoke prefilter: {before} -> {len(hdf)} rows by mirrored "
          f"patient_id set ({len(mirrored)} mirrored patients).")
    hdf["_exists"] = hdf["image_path"].apply(
        lambda p: os.path.exists(os.path.join(BASE_IMAGE_DIR, str(p)))
    )
    hdf = hdf[hdf["_exists"]].drop(columns=["_exists"]).head(MAX_SAMPLES)
    print(f"smoke prefilter: kept {len(hdf)} rows with on-disk images.")
    ds = DSCls(**ds_kwargs, harmonized_df=hdf)

## Build the evaluator

In [ ]:
from radharmony.evaluator import ReportGenerationEvaluator
from radharmony.evaluator.metrics import FULL_METRICS, LIGHT_METRICS

ev = ReportGenerationEvaluator(
    report_generator,
    dataset=ds,
    metrics=LIGHT_METRICS if LIGHT else FULL_METRICS,
    ref_section=ref_section,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    max_samples=MAX_SAMPLES,
    n_bootstrap=N_BOOTSTRAP,
    output_dir=OUTPUT_DIR,
)

## Run

**`MODE == "generate"`** (the two-env path, e.g. CheXagent-2): generate and
dump the pairs parquet only — no RadEval import. Then score the parquet with
[`score_reports_radeval.ipynb`](score_reports_radeval.ipynb).

**`MODE == "both"`**: generate **and** score in one environment — only valid
when a single venv satisfies both the backbone and RadEval.

In [ ]:
import os

if MODE == "generate":
    pairs = PAIRS_PARQUET or os.path.join(OUTPUT_DIR, "pairs.parquet")
    path = ev.generate_only(pairs)
    print(f"Generated {ev.n_samples_} (id, ref, hyp) pairs -> {path}")
    print("Stage B: open score_reports_radeval.ipynb and set "
          f"PAIRS_PARQUET = {path!r}")
else:  # both
    df = ev.evaluate()
    out = ev.save_results(df)
    print(f"Scored {ev.n_samples_} report pairs -> {out}/")
    df